# Marketplace Analytics — exploratory research

This notebook uses generated **fictional demo data**. Results demonstrate methods, not real business findings.

**Questions:** Which months lead GMV? How many buyers return? How do delayed deliveries relate to review scores?

**Methodological constraints:** observational data; eligible orders exclude canceled/unavailable; GMV excludes freight.


## Project setup


In [ ]:
from marketplace_analytics.config import DB
from marketplace_analytics.analytics import get_orders, kpis, monthly_sales, cohort_retention, rfm_segments, delivery_summary
import pandas as pd
import matplotlib.pyplot as plt
from marketplace_analytics.bootstrap import ensure_demo_database
ensure_demo_database()
orders = get_orders(DB)
print(f"Loaded {len(orders):,} order-grain rows")
orders.head()

## KPI exploration


In [ ]:
pd.Series(kpis(orders)).to_frame("value")

## Monthly sales


In [ ]:
monthly = monthly_sales(orders)
monthly.plot(x="order_month", y="gmv", kind="line", marker="o", figsize=(11,4), title="Monthly GMV (BRL)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Buyer cohort retention


In [ ]:
retention = cohort_retention(orders)
retention.head(15)

## Retention heatmap


In [ ]:
matrix = retention[retention.period.le(11)].pivot(index="cohort", columns="period", values="retention")
plt.figure(figsize=(12,6))
plt.imshow(matrix.fillna(-0.1), aspect="auto", vmin=0, vmax=1, cmap="Blues")
plt.yticks(range(len(matrix.index)), matrix.index)
plt.xticks(range(len(matrix.columns)), matrix.columns)
plt.colorbar(label="Retention share")
plt.title("Monthly buyer retention (blank=unobserved)")
plt.xlabel("Months since first purchase")
plt.tight_layout()
plt.show()

## Delivery performance and reviews


In [ ]:
delivery_summary(orders)

## RFM customer segmentation


In [ ]:
rfm = rfm_segments(orders)
rfm.groupby("segment").agg(buyers=("customer_unique_id", "count"), gmv=("monetary", "sum")).sort_values("buyers", ascending=False)

## Interpretation and recommendations

1. Check monthly GMV and order counts together; GMV changes can reflect AOV or volume.
2. Evaluate repeated purchase cohorts by observation window; late cohorts are right-censored.
3. Compare customer review distributions across on-time/late deliveries, **without interpreting correlation as causal**.
4. Treat RFM groups as marketing hypotheses, not proof of uplift. For impact, propose a randomized intervention.
5. Label all numerical findings as synthetic demo results in your portfolio.
